# 04. 경로 틀이 둘이면 이름으로는 알 수 없다

> 2026-09-04 · 이동원 · 결론 문서:
> [기능명세 v1.4](../../docs/기능명세/version1.4/공공데이터포털_경로틀과_예탁결제원.md)

어제(09-03) 조사 세션이 "못 찾음" 으로 남아 있던 포털 서비스 6개를 전부 찾았습니다.
경로 조합이 틀린 게 아니라 **틀 자체가 둘**이었습니다. 오늘은 그 결과를 코드로 옮기고,
바꾼 코드가 실제 서버와 맞는지 실호출 5건으로 확인했습니다.

이 노트북은 **외부 API 를 부르지 않습니다.** 코드 상수와 어제 받아 둔 응답 원문(문자열로
옮김)으로 셋을 보입니다.

| 무엇 | 이전 | 지금 |
|---|---|---|
| `BASE_URL` | `…/1160100/service` (구세대 틀이 박혀 있었다) | 호스트만 |
| 엔드포인트 | 2개 | **17개** · 상수가 경로 전체를 갖는다 |
| 날짜 자리표시자 | 바닥 `00010101` 만 | + 천장 **`99991231`** |
| 예탁결제원 | 없음 | `ksd_data.py` — XML · 선언 파라미터만 · 상한 200 |

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))
import pandas as pd

from ingest.clients import data_go_kr as dgk
from ingest.clients import ksd_data as ksd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

## 1. 같은 `_V2` 인데 한쪽은 `/service/` 가 있고 한쪽은 없다

`GetCorpBasicInfoService_V2` 는 구세대(`/service/` 있음), `GetStocDiviInfoService_V2` 는
신세대(없음)입니다. 접미사로도 이름 모양으로도 세대를 알 수 없습니다. 그래서 규칙으로
계산하지 않고 **상수가 경로 전체를 말하게** 했습니다.

In [2]:
표 = pd.DataFrame(
    [(이름, dgk.endpoint_generation(경로), 경로) for 이름, 경로 in dgk.ENDPOINTS.items()],
    columns=["이름", "세대", "경로 (호스트 뒤)"],
)
print(f"엔드포인트 {len(표)}개 · 구세대 {(표['세대'] == 'legacy').sum()} · "
      f"신세대 {(표['세대'] == 'modern').sum()}")
표

엔드포인트 17개 · 구세대 6 · 신세대 11


,이름,세대,경로 (호스트 뒤)
0,listed,legacy,1160100/service/GetKrxListedInfoService/getItemInfo
1,corp_outline,legacy,1160100/service/GetCorpBasicInfoService_V2/getCorpOutline_V2
2,gov_shareholder,legacy,1160100/service/GetCorpGoveInfoService/getStockholderInfo
3,gov_ceo,legacy,1160100/service/GetCorpGoveInfoService/getReprDireInfo
4,gov_exec_pay,legacy,1160100/service/GetCorpGoveInfoService/getExecRemuStat
5,gov_executives,legacy,1160100/service/GetCorpGoveInfoService/getExecutivesInfo
6,item_basic,modern,1160100/GetStocIssuInfoService_V3/getItemBasiInfo_V3
7,issue_stat,modern,1160100/GetStocIssuInfoService_V3/getStocIssuStat_V3
8,issue_history,modern,1160100/GetStocIssuInfoService_V3/getStocIssuInfo_V3
9,lockup,modern,1160100/GetStocIssuInfoService_V3/getLockUpRetuInfo_V3


In [3]:
# 같은 _V2 접미사인데 세대가 갈린다 — 이름으로 유추하면 여기서 틀린다.
둘 = 표[표["경로 (호스트 뒤)"].str.contains("_V2/")]
둘[["이름", "세대", "경로 (호스트 뒤)"]]

,이름,세대,경로 (호스트 뒤)
1,corp_outline,legacy,1160100/service/GetCorpBasicInfoService_V2/getCorpOutline_V2
10,dividend,modern,1160100/GetStocDiviInfoService_V2/getDiviInfo_V2
11,rights_schedule,modern,1160100/GetStocRighScheService_V2/getRighExerReasSche_V2
12,lend_by_item,modern,1160100/GetStocLendBorrInfoService_V2/getStItemLendAndBorrStatu_V2
13,lend_by_participant,modern,1160100/GetStocLendBorrInfoService_V2/getStBusiTypePartStatu_V2
14,distribution,modern,1160100/GetStocTradInfoService_V2/getStocDistInfo_V2
15,irregular_stock,modern,1160100/GetStocTradInfoService_V2/getIrreRigforSecu_V2
16,deposit_available,modern,1160100/GetStocDepoInfoService_V2/getDepoAvaiWhet_V2


## 2. 바닥선만 있으면 천장으로 새어 들어온다

`00010101`(서기 1년 · "해당 없음")은 이미 거르고 있었습니다. 어제 반대쪽을 만났습니다 —
`99991231` 이 "아직 폐지 안 됨" 입니다. 상장폐지일·법인 소멸일·만기일 자리에 옵니다.
그대로 두면 "9999년에 상장폐지되는 회사" 가 생기고 잔여기간 계산이 **에러 없이** 틀어집니다.

In [4]:
표본 = ["00010101", "11111111", "18970925", "76/03/24", "2025/08/07",
        "21001231", "99991231", "9999/12/31", ""]
pd.DataFrame({"원문": 표본, "normalize_date": [dgk.normalize_date(x) for x in 표본],
              "뜻": ["해당 없음 (자리표시자)", "쓰레기값", "동화약품 창업 — 진짜",
                    "두 자리 연도 → 1976", "YYYY/MM/DD", "천장 — 통과",
                    "아직 폐지 안 됨 (자리표시자)", "같은 값 · 구분자만 다름", "빈 값"]})

,원문,normalize_date,뜻
0,00010101,NaN,해당 없음 (자리표시자)
1,11111111,NaN,쓰레기값
2,18970925,18970925,동화약품 창업 — 진짜
3,76/03/24,19760324,두 자리 연도 → 1976
4,2025/08/07,20250807,YYYY/MM/DD
5,21001231,21001231,천장 — 통과
6,99991231,NaN,아직 폐지 안 됨 (자리표시자)
7,9999/12/31,NaN,같은 값 · 구분자만 다름
8,,NaN,빈 값


## 3. 예탁결제원은 규약이 셋 다 다르다

같은 키가 통하니 같은 클라이언트를 쓰고 싶어집니다. 그런데 **XML 전용 · 선언 안 된
파라미터 거절 · `numOfRows` 상한 200** 이라 따로 두었습니다. 아래는 어제 실제로 받은
응답 원문입니다(`probe_out/` · 삼성전자 발행회사번호 593).

In [5]:
# 선언 안 된 파라미터(numOfRows·pageNo)를 붙였을 때 실제로 온 것 — body 가 없다.
거절 = '''<response><header><resultCode>10</resultCode>
<resultMsg>INVALID_REQUEST_PARAMETER_ERROR</resultMsg></header></response>'''

# 그래서 부르기 **전에** 세운다 — 예산을 깎지 않는다.
try:
    ksd._build_url(ksd.operation("issuer_stock_changes"), "키",
                   {"issucoCustno": "593", "numOfRows": 5})
except ksd.KsdError as e:
    print(str(e).splitlines()[0])

CorpSvc/getIssucoStkQtyChgList 가 받지 않는 파라미터다: ['numOfRows']


In [6]:
# 제대로 부른 응답 — 액면분할이 사유코드와 발행일로 온다.
주식수변동 = '''<response>
  <header><resultCode>00</resultCode><resultMsg>NORMAL_SERVICE</resultMsg></header>
  <body><items>
    <item><issuDt>20180503</issuDt><secnIssuRacd>201</secnIssuRacd>
      <secnIssuRacdNm>액면분할</secnIssuRacdNm><issuQty>6419324700</issuQty>
      <listDt>20180504</listDt></item>
    <item><issuDt>20180503</issuDt><secnIssuRacd>201</secnIssuRacd>
      <secnIssuRacdNm>액면분할</secnIssuRacdNm><issuQty>903629000</issuQty>
      <listDt>20180504</listDt></item>
    <item><issuDt>20120406</issuDt><secnIssuRacd>207</secnIssuRacd>
      <secnIssuRacdNm>합병</secnIssuRacdNm><issuQty>269867</issuQty><listDt></listDt></item>
  </items><totalCount>3</totalCount></body>
</response>'''
루트 = ksd._parse(주식수변동, "CorpSvc/getIssucoStkQtyChgList")
pd.DataFrame([ksd.parse_qty_change(r) for r in ksd._items(루트)])

,issu_dt,reason_code,reason_nm,issu_qty,list_dt
0,20180503,201,액면분할,6419324700,20180504
1,20180503,201,액면분할,903629000,20180504
2,20120406,207,합병,269867,NaN


In [7]:
# 기업개요의 dlistDt·custXtinDt 가 99991231 — 천장이 None 으로 바꾼다.
개요 = ksd.parse_basic_info({
    "issucoCustno": "593", "shotnIsin": "005930", "repSecnNm": "삼성전자",
    "founDt": "19690113", "apliDt": "19750611", "dlistDt": "99991231",
    "custXtinDt": "99991231", "caltotMartTpcd": "11", "pval": "100",
    "totalStkCnt": "   6,648,649,811 주", "eltscYn": "Y",
})
pd.Series(개요).to_frame("값")

,값
issuco_custno,593
code,005930
name,삼성전자
bizno,None
ceo,None
found_dt,19690113
market_code,11
market,유가증권시장
list_dt,19750611
delist_dt,None


In [8]:
# 상한 200 — 콜 수가 금융위(1,000)보다 5배 든다.
pd.DataFrame({
    "행 수": [943, 2_728, 15_913],
    "금융위 콜 (1,000/쪽)": [-(-n // dgk.PAGE_SIZE) for n in (943, 2_728, 15_913)],
    "예탁결제원 콜 (200/쪽)": [ksd.estimate_calls(n) for n in (943, 2_728, 15_913)],
}, index=["유가 종목 수", "하루 상장종목", "종목기본정보 하루"])

,행 수,"금융위 콜 (1,000/쪽)",예탁결제원 콜 (200/쪽)
유가 종목 수,943,1,5
하루 상장종목,2728,3,14
종목기본정보 하루,15913,16,80


## 4. 실호출 5건 — 바꾼 코드가 서버와 맞는가 (2026-09-04 · 예산 5콜)

시험 122건은 우리가 적은 응답으로 돕니다. 서버와 맞는지는 실호출로만 알 수 있습니다.
이 노트북은 부르지 않고 결과만 적습니다.

| 무엇 | 호출 | 결과 |
|---|---|---|
| 구세대 경로가 여전히 도나 | `page_count(EP_LISTED, basDt=20240830)` | **3쪽** ✅ |
| 신세대 `_V3` 경로 | `page_count(EP_ITEM_BASIC, basDt=20240830)` | **16쪽** ✅ |
| KSD 다리 | `issuer_custno("KR7005930003")` | **`593`** ✅ |
| KSD 주식수변동 | `stock_qty_changes("593")` | 5행 · 20180503 액면분할 6,419,324,700 ✅ |
| KSD 기업개요 | `issuer_basic_info("593")` | `delist_dt=None` · `total_shares=6,648,649,811` ✅ |

## 5. 참조 자료 반출 — 정본 스크립트가 어제와 같은 것을 만든다

어제 스크래치패드 스크립트로 올린 `identity/`·`financial/`·`macro/`·`calendar/` 를
`scripts/export_reference_dataset.py` 로 옮겼습니다. 오늘 돌린 결과입니다.

In [9]:
import json

경로 = (Path.cwd().parent.parent / "data" / "outbox" / "reference_20260904"
        / "MANIFEST_reference.json")
if 경로.exists():
    m = json.loads(경로.read_text("utf-8"))
    print(f"만든 시각 {m['generated_at']} · 경계 {m['holdout_start']} "
          f"({m['holdout_start_authority']})")
    display(pd.DataFrame([{k: f[k] for k in ("path", "rows", "size_mb", "range")}
                          for f in m["files"]]))
else:
    print("반출본이 없다 — python scripts/export_reference_dataset.py 를 먼저 돌린다")

만든 시각 2026-09-04T14:25:37+09:00 · 경계 20240901 (evaluation/horizon.py HOLDOUT_START)


,path,rows,size_mb,range
0,identity/stock_identity_dev.parquet,2871366,5.990,"[20200102, 20240830]"
1,identity/corp_profile_dev.parquet,22137,0.370,"[20200509, 20240830]"
2,financial/dart_financial_dev.parquet,498059,8.940,"[20150923, 20240814]"
3,macro/macro_series_dev.parquet,15782,0.114,"[20090828, 20240831]"
4,calendar/trading_calendar_dev.parquet,10854,0.039,"[20100104, 20240830]"


## 6. 오늘 배운 것

**이름은 근거가 아니다. 실호출이 근거다.** 어제는 종목명으로 우선주를 판정하면 개명 때
틀린다는 것을 배웠고, 오늘은 서비스명으로 경로를 유추하면 애초에 못 찾는다는 쪽이었다.

**한쪽 자리표시자를 만나면 반대쪽도 있는지 묻는다.** 바닥선만 두면 천장으로 새어 든다.

**정본이 없으면 재현이 없다.** 잘 올라갔고 SHA 대조까지 했어도, 만든 스크립트가 저장소에
없으면 다음에 같은 것을 만들 수 없다.

> [TIL 2026-09-04](../../docs/TIL/이동원/2026-09-04-경로-틀이-둘이면-이름으로는-알-수-없다.md)